# QPE Default Calibration

This notebook calibrates **QPE baseline defaults**.

Unlike VQE and VarQITE, QPE settings are often strongly problem-dependent. This notebook is therefore best used to choose:

- a sensible tutorial/default configuration for small molecules
- a documented calibration path for larger systems

The sweep covers:

- ancilla count
- evolution time `t`
- Trotter steps
- shot count
- seed sensitivity

By default it targets `H2`, because that is the smallest molecule where QPE calibration is still instructive.

## How To Use This Notebook

- Edit the calibration grids in the setup cell rather than changing the ranking logic.
- Use this notebook to choose baseline defaults for small molecules, not to assume one globally optimal QPE configuration exists.
- Keep multiple seeds whenever `shots` is finite so the ranking includes sampling variability.
- Use `recommended_defaults` for default-friendly settings and `top_candidates` for the absolute best settings in the tested grid.


In [ ]:
from __future__ import annotations

import time
from contextlib import redirect_stdout
from io import StringIO
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from common.hamiltonian import get_exact_spectrum
from qpe import run_qpe

MOLECULE = "H2"
MAPPING = "jordan_wigner"
ANCILLA_GRID = [2, 3, 4]
TIME_GRID = [0.5, 1.0, 2.0]
TROTTER_GRID = [1, 2]
SHOT_GRID = [500, 1000, 2000]
SEEDS = [0, 1, 2]
EXACT_GROUND = get_exact_spectrum(MOLECULE, k=1)[0]


def timed_call(fn, /, **kwargs):
    t0 = time.perf_counter()
    sink = StringIO()
    with redirect_stdout(sink):
        result = fn(**kwargs)
    return result, time.perf_counter() - t0


def stats(xs):
    arr = np.asarray(xs, dtype=float)
    return {
        "mean": float(arr.mean()),
        "std": float(arr.std(ddof=0)),
        "min": float(arr.min()),
        "max": float(arr.max()),
    }


TOTAL_RUNS = len(ANCILLA_GRID) * len(TIME_GRID) * len(TROTTER_GRID) * len(SHOT_GRID) * len(SEEDS)
PROGRESS_EVERY = max(1, TOTAL_RUNS // 12)


In [ ]:
records = []
run_index = 0


for ancillas in ANCILLA_GRID:
    for t in TIME_GRID:
        for trotter_steps in TROTTER_GRID:
            for shots in SHOT_GRID:
                for seed in SEEDS:
                    run_index += 1
                    if run_index == 1 or run_index % PROGRESS_EVERY == 0 or run_index == TOTAL_RUNS:
                        print(
                            f"[{run_index}/{TOTAL_RUNS}] "
                            f"ancillas={ancillas} | t={t} | trotter={trotter_steps} | "
                            f"shots={shots} | seed={seed}"
                        )
                    result, runtime_s = timed_call(
                        run_qpe,
                        molecule=MOLECULE,
                        mapping=MAPPING,
                        n_ancilla=ancillas,
                        t=t,
                        trotter_steps=trotter_steps,
                        shots=shots,
                        seed=seed,
                        noisy=False,
                        plot=False,
                        force=False,
                    )
                    records.append(
                        {
                            "ancillas": int(ancillas),
                            "t": float(t),
                            "trotter_steps": int(trotter_steps),
                            "shots": int(shots),
                            "seed": int(seed),
                            "energy": float(result["energy"]),
                            "phase": float(result["phase"]),
                            "hf_energy": float(result["hf_energy"]),
                            "abs_error": abs(float(result["energy"]) - EXACT_GROUND),
                            "runtime_s": float(runtime_s),
                            "num_qubits": int(result["num_qubits"]),
                            "total_wires": int(result["num_qubits"] + ancillas),
                        }
                    )

records_df = pd.DataFrame(records)
print(f"Completed {len(records_df)} QPE calibration runs.")
display(records_df.head(10).round(6))


In [ ]:
grouped = defaultdict(list)
for row in records:
    key = (row["ancillas"], row["t"], row["trotter_steps"], row["shots"])
    grouped[key].append(row)

summary = []
for key, rows in sorted(grouped.items()):
    errors = [r["abs_error"] for r in rows]
    runtimes = [r["runtime_s"] for r in rows]
    energies = [r["energy"] for r in rows]
    summary.append(
        {
            "ancillas": key[0],
            "t": key[1],
            "trotter_steps": key[2],
            "shots": key[3],
            "mean_abs_error": stats(errors)["mean"],
            "std_abs_error": stats(errors)["std"],
            "mean_energy": stats(energies)["mean"],
            "mean_runtime_s": stats(runtimes)["mean"],
            "score": stats(errors)["mean"] + 0.25 * stats(errors)["std"] + 0.0005 * stats(runtimes)["mean"],
        }
    )

summary_df = pd.DataFrame(summary).sort_values(["score", "mean_abs_error", "mean_runtime_s"])
display(summary_df.round(8))


## How To Interpret The Results

- `top_candidates` lists the strongest settings in the tested grid, regardless of whether they are expensive.
- `recommended_defaults` filters that space down to settings that are more realistic as user-facing defaults.
- If a larger-system molecule disagrees strongly with the H2 ranking, keep the H2 result as a documented baseline default rather than pretending the default is globally optimized.


In [ ]:
top_candidates = sorted(summary, key=lambda row: (row["score"], row["mean_abs_error"], row["mean_runtime_s"]))[:15]
top_candidates_df = pd.DataFrame(top_candidates)
print("Top QPE candidates")
display(top_candidates_df.round(8))


In [ ]:
default_friendly = [row for row in summary if row["ancillas"] <= 4 and row["shots"] <= 2000 and row["trotter_steps"] <= 2]
recommended_defaults = sorted(
    default_friendly,
    key=lambda row: (row["score"], row["mean_abs_error"], row["mean_runtime_s"]),
)[:10]
recommended_defaults_df = pd.DataFrame(recommended_defaults)
print("Recommended QPE default candidates")
display(recommended_defaults_df.round(8))


In [ ]:
rows = top_candidates[:8]
labels = [f"a={row['ancillas']}\nt={row['t']}\nr={row['trotter_steps']}\nshots={row['shots']}" for row in rows]
values = [row["mean_abs_error"] for row in rows]

plt.figure(figsize=(10, 4))
plt.bar(labels, values, alpha=0.85)
plt.ylabel("Mean absolute error (Ha)")
plt.title(f"Top QPE calibration candidates for {MOLECULE}")
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()